# DEGISKENLER → DEGISKENLER_COMPACT

1040 satırlık kolon sözlüğünü, LLM promptuna gömülebilir ~240 satırlık compact
sözlüğe indirger. Kolon adları desenlidir (`AILE_METRIK`); aileler isimden
çıkarılır, doğruluğu `aciklama` kolonuna karşı **5 eksenli denetimle** kanıtlanır.
Denetim geçmezse pipeline **HATA verir (assert)** — yanlış sözlük asla yazılmaz.

**Akış:** kütüphaneler → config → kurallar → veri yükleme → gruplama → denetim → final dataset


## 1. Kütüphaneler

In [ ]:
import re
from collections import defaultdict

import dataiku
import pandas as pd


## 2. Konfigürasyon
Girdi/çıktı dataset adları ve sözlük tablosundaki kolon adları.
Başka bir projede yeniden kullanırken yalnızca bu hücre değişir.

In [ ]:
DICT_DATASET = "DEGISKENLER"          # kolon sözlüğü tablosu
NAME_COL     = "SM_ID"                # kolon-adı alanı
DESC_COL     = "aciklama"             # açıklama alanı
OUT_DATASET  = "DEGISKENLER_COMPACT"  # üretilecek compact sözlük dataset'i


## 3. Lejant ve aile ayrıştırma kuralları
- Lejantın hakemi her zaman `aciklama` kolonudur; tanımlar oradan alınmıştır.
- `X2` = aynı uzunlukta bir **önceki** pencere (`3D_X2` = 3-6 gün arası).
- `OUT_`/`IN_` kolonları (karşı taraf analitiği): **aile = ilk 2 parça**, metrik = kalan.
- `TXN_` kolonları: metrik son eki **sondan** soyulur (uzun eşleşme önce).

In [ ]:
ABBREVIATION_LEGEND = {
    "TXN": "işlem", "GDN": "giden para hareketi", "GLN": "gelen para hareketi",
    "OUT": "giden işlemlerde karşı taraf analitiği",
    "IN": "gelen işlemlerde karşı taraf analitiği",
    "CP": "karşı taraf (counterparty)",
    "KRIPTO": "kripto şirketleri (karşı taraf segmenti; GDN ile = kripto'ya gönderilen para)",
    "ODEME": "ödeme şirketleri/kuruluşları (karşı taraf segmenti)",
    "BAHIS": "bahis şirketleri (karşı taraf segmenti; GDN ile = bahis harcaması)",
    "3D/7D/14D/30D/60D/90D/180D/360D": "son N gün penceresi",
    "X2": "aynı uzunlukta bir önceki pencere (ör. 3D_X2 = 3-6 gün arası)",
    "3D_6D": "3-6 gün arası (önceki dönem)",
    "H00_06 / H06_12 / H12_18 / H18_24": "günün saat dilimi",
    "HS": "hafta sonu",
    "ACTIVE_MONTH": "aktif ay sayısı (o kanalda işlem görülen ay adedi)",
    "CNT": "işlem adedi", "AMT": "işlem tutarı",
    "AMT_AVG": "ortalama tutar", "AMT_MAX": "maksimum tutar",
    "AMT_STD": "tutar standart sapması", "CV": "tutar değişim katsayısı",
    "PER_DAY": "günlük ortalama", "BNK_ADT": "farklı banka adedi",
    "SHR": "pay (o pencerede ilgili segmentin toplam içindeki oranı)",
    "RATIO": "iki pencere oranı (kısa pencere / uzun pencere, ör. 3D/7D)",
    "TMSNCFRST": "penceredeki İLK işlemden bu yana geçen süre",
    "TMSNCLST": "penceredeki SON işlemden bu yana geçen süre",
    "TOP1_CP": "en çok işlem yapılan karşı taraf",
    "TOTAL_CP": "tüm karşı taraflar toplamı",
    "DISTINCT_CP": "farklı karşı taraf sayısı",
    "EFFECTIVE_CP_BY_TXN/AMT": "işlem/tutar bazında etkin karşı taraf sayısı",
    "HHI": "yoğunlaşma endeksi (Herfindahl)",
    "ENTROPY": "dağılım entropisi",
    "RATIO_RAW / RATIO_SHRUNK": "TOP1 payı: ham / düzeltilmiş (shrink)",
    "SUPPORT_SCORE": "destek skoru", "CONC_SCORE": "yoğunlaşma skoru",
    "LOW_SUPPORT_FLAG": "düşük destek bayrağı",
    "NO_TXN_FLAG / NO_AMT_FLAG": "o pencerede işlem/tutar yok bayrağı",
    "HIGH_CONC_FLAG": "yüksek yoğunlaşma bayrağı",
    "GCK": "gecikme", "ADT": "adet", "BKY": "bakiye", "LMT": "limit", "KRD": "kredi",
}

# TXN_* kolonları için metrik son ekleri — uzunluğa göre otomatik sıralanır
# (uzun önce): CNT_SHR/CNT, AMT_RATIO/AMT gibi çakışmalar böylece imkansız.
NEW_SUFFIXES = sorted({
    "AMT_PER_DAY", "CNT_PER_DAY", "AMT_AVG", "AMT_MAX",
    "HS_CNT", "HS_AMT", "BNK_ADT", "CNT", "AMT",
    "HS_CNT_SHR", "HS_AMT_SHR", "CNT_SHR", "AMT_SHR",
    "CNT_RATIO", "AMT_RATIO",
    "AMT_STD", "CV",
    "TMSNCFRST", "TMSNCLST",
}, key=len, reverse=True)


def family_of(col: str, suffixes=NEW_SUFFIXES):
    """Kolonu (aile, metrik) olarak ayırır; eşleşmeyen 'özel kolon' sayılır.
    1) OUT_/IN_ önekli CP kolonları: aile = ilk 2 parça (OUT_7D),
       metrik = kalan her şey (TOP1_CP_TXN_RATIO_SHRUNK gibi).
    2) Diğerleri (TXN_*): metrik suffix'i sondan soyulur."""
    parts = col.split("_")
    if parts[0] in ("OUT", "IN") and len(parts) >= 3:
        return "_".join(parts[:2]), "_".join(parts[2:])
    for suf in suffixes:
        if col.endswith("_" + suf):
            return col[: -(len(suf) + 1)], suf
    return None, None


## 4. Veri yükleme
Sözlük tablosu okunur; kolon adları ve değerler trim'lenir.

In [ ]:
df = dataiku.Dataset(DICT_DATASET).get_dataframe()
df.columns = [c.strip() for c in df.columns]

print(f"{DICT_DATASET}: {len(df)} satır, kolonlar: {list(df.columns)}")


## 5. Gruplama
Her kolon ya bir aileye `(metrik, açıklama)` olarak ya da özel kolon listesine düşer.
Beklenen özel kolonlar: `PERIOD` ve target (`TGT_*`).

In [ ]:
families = defaultdict(list)   # aile -> [(metrik, aciklama)]
specials = []                  # desene uymayanlar

for _, row in df.iterrows():
    sm_id = str(row[NAME_COL]).strip()
    desc = str(row[DESC_COL]).strip()
    fam, metric = family_of(sm_id)
    if fam:
        families[fam].append((metric, desc))
    else:
        specials.append((sm_id, desc))

print(f"{len(df)} kolon -> {len(families)} aile + {len(specials)} özel kolon")
print("Özel kolonlar:", [s for s, _ in specials])


## 6. Beş eksenli denetim (geçmezse pipeline durur)
1. **Kayıpsızlık:** aile+metrik birleşimi orijinal adları birebir geri kurar.
2. **Çift sayım yok:** her kolon tam 1 kez atanır.
3-4. **Anlam + sayı:** metrik parçaları ve addaki sayılar (`X2`/`TOP1`/`HXX` kurallı)
   açıklamayla birebir tutarlıdır; kuralı tanımsız token kabul edilmez.
5. **Çekirdek homojenliği:** bir ailenin tüm üyeleri aynı yön/segment/sayı kümesini anlatır.

In [ ]:
METRIC_KEYWORDS = {
    "CNT": ["adet", "adedi", "sayısı"], "AMT": ["tutar"],
    "AVG": ["ortalama"], "MAX": ["maksimum"], "STD": ["standart"],
    "CV": ["katsayısı", "cv"], "PER": ["günlük"], "DAY": ["günlük"],
    "BNK": ["banka"], "ADT": ["adet", "adedi"], "SHR": ["pay"],
    "RATIO": ["oran"], "TMSNCFRST": ["ilk"], "TMSNCLST": ["son"],
    "HS": ["hafta sonu"], "ACTIVE": ["ay"], "MONTH": ["ay"],
    "TOP1": ["en çok"], "TOTAL": ["toplam"], "DISTINCT": ["farklı"],
    "EFFECTIVE": ["etkin"], "CP": ["karşı taraf"],
    "HHI": ["hhi", "yoğunlaş"], "ENTROPY": ["entropi"],
    "SUM": ["tutar"], "RAW": ["ham"], "SHRUNK": ["düzeltilmiş", "shrink"],
    "SUPPORT": ["destek"], "CONC": ["yoğunlaş"], "SCORE": ["skor"],
    "FLAG": ["bayrağı", "bayrak"], "NO": ["yok"], "LOW": ["düşük"],
    "HIGH": ["yüksek"], "TXN": ["işlem"], "X2": ["önceki"], "BY": [],
}


def expected_numbers(col: str) -> set:
    """Kolon adından açıklamada BEKLENEN sayı kümesini üretir.
    ND pencereler -> N; X2 varsa her pencere için 2N eklenir;
    HXX/XX saat token'ları -> saat sayıları; TOP1'in 1'i YOK sayılır."""
    toks = col.split("_")
    windows, hours, double = set(), set(), False
    for t in toks:
        if t == "TOP1":
            continue
        if t == "X2":
            double = True
            continue
        m = re.fullmatch(r"(\d+)D", t)
        if m:
            windows.add(int(m.group(1)))
            continue
        m = re.fullmatch(r"H?(\d+)", t)
        if m:
            hours.add(int(m.group(1)))
    if double:
        windows |= {2 * n for n in set(windows)}
    return windows | hours


def core_of(desc: str):
    """Aile çekirdeği: (yön, segment kümesi, açıklamadaki sayı kümesi)."""
    d = desc.lower()
    return (("giden" in d, "gelen" in d),
            tuple(s for s in ("kripto", "ödeme", "bahis") if s in d),
            tuple(sorted({int(x) for x in re.findall(r"\d+", desc)})))


# --- Eksen 1+2: kayıpsızlık ve çift sayım ---
reconstructed = {f"{fam}_{m}" for fam, items in families.items() for m, _ in items}
reconstructed |= {s for s, _ in specials}
originals = {str(x).strip() for x in df[NAME_COL]}
total_assigned = sum(len(it) for it in families.values()) + len(specials)
assert reconstructed == originals, f"Kayıp/hayalet: {originals ^ reconstructed}"
assert total_assigned == len(df), f"Çift sayım: {total_assigned} != {len(df)}"

# --- Eksen 3+4: metrik anlamı + tam sayı tutarlılığı ---
violations, unaudited = [], defaultdict(int)
for fam, items in families.items():
    for metric, desc in items:
        col, d = f"{fam}_{metric}", desc.lower()
        got = {int(x) for x in re.findall(r"\d+", desc)}
        if expected_numbers(col) != got:
            violations.append((col, "SAYI", desc))
        for tok in metric.split("_"):
            if tok.isdigit() or re.fullmatch(r"\d+D", tok) or re.fullmatch(r"H\d+", tok):
                continue
            if tok in METRIC_KEYWORDS:
                kws = METRIC_KEYWORDS[tok]
                if kws and not any(k in d for k in kws):
                    violations.append((col, f"METRİK:{tok}", desc))
            else:
                unaudited[tok] += 1
assert not violations, f"{len(violations)} ihlal, ilk 5: {violations[:5]}"
assert not unaudited, f"Denetim dışı token: {dict(unaudited)}"

# --- Eksen 5: aile çekirdeği homojenliği ---
hetero = [f for f, it in families.items() if len({core_of(d) for _, d in it}) > 1]
assert not hetero, f"Çekirdek tutarsız aileler: {hetero[:10]}"

print("✓ 5 eksenli denetim geçti — sözlük kanıtla doğrulandı.")


## 7. Final: compact sözlüğü derle ve dataset'e yaz
Çıktı tek satırlık dataset (`part` + `text`). Sonraki adım (LLM döngü recipe'si)
bu metni okuyup prompta gömer. Aile satırındaki örnek açıklama sadece navigasyon
içindir; nihai anlam her zaman DEGISKENLER'deki **orijinal** açıklamadan alınır.

In [ ]:
lines = ["# Veri Sözlüğü (Compact) — DEGISKENLER", ""]

lines.append("## ÖZEL KOLONLAR (birebir)")
for sm_id, desc in specials:
    lines.append(f"- `{sm_id}`: {desc}")

lines += ["", "## KISALTMA LEJANTI"]
for k, v in ABBREVIATION_LEGEND.items():
    lines.append(f"- `{k}`: {v}")

lines += ["", "## KOLON AİLELERİ (aile + mevcut metrikler + örnek açıklama)"]
for fam in sorted(families):
    metrics = sorted({m for m, _ in families[fam]})
    ornek = families[fam][0][1]
    lines.append(f"- `{fam}_[{'|'.join(metrics)}]` — örn: {ornek}")

compact_text = "\n".join(lines)
print(f"Compact: {len(compact_text)} karakter (~{len(compact_text)//3} token), "
      f"{len(families)} aile + {len(specials)} özel kolon")

out = pd.DataFrame({"part": ["compact_dictionary"], "text": [compact_text]})
dataiku.Dataset(OUT_DATASET).write_with_schema(out)
print(f"{OUT_DATASET} yazıldı ✓")
